# Gradio

- 파이썬 함수, ML 모델, API를 손쉽게 웹 데모나 앱 형태로 만들 수 있는 오픈소스 라이브러리.
- 별도의 JS, CSS, 서버 지식 없이 간단한 코드로 UI 작성 및 공유 링크 생성 가능.

## Gradio 설치

```bash
pip install --upgrade gradio
```


## Gradio UI 

### 입출력 구분 화면
  - `with gr.Blocks() as demo:` Gradio의 기본 **앱 컨테이너(Blocks)** 를 demo라는 alias로 생성
  - `gr.Markdown(WELCOME_MESSAGE)` 화면에 안내 문구를 보여주는 Markdown 
  - `with gr.Row(): / with gr.Column():` 레이아웃 구성 - Row는 가로 방향 줄, Column은 세로 방향 칸
  - `gr.Textbox(...)` 글자를 입력하거나 결과를 출력하는 텍스트 상자 - label과 편집 속성 interactive=False 설정
  - `gr.Button("입력")` 버튼
  - `submit_btn.click(...)` 버튼 클릭 이벤트 처리
  - `demo.launch()` 앱 실행 - 웹 브라우저에서 접근 가능
  - `demo.close()` 앱 정지 - 서버를 close 하지 않는 경우 새로운 port로 추가 생성

In [ ]:
# 필요한 모듈 임포트
import gradio as gr

In [ ]:

def show_character_info(user_input: str) -> str:
    """입력된 캐릭터 이름을 받아 안내 문구를 반환한다."""

    message = f"{user_input}에 대해 알아 봅시다"
    return message

with gr.Blocks() as demo:

    gr.Markdown("""
        ## 🕵️‍♂️ 명탐정 코난 세계관에 오신 것을 환영합니다.
        ### 명탐정 코난의 등장 인물 중 좋아하는 캐릭터를 입력하세요
    """)

    with gr.Row():
        with gr.Column(scale=1):
            character_input = gr.Textbox(label="캐릭터 이름 입력")
            submit_btn = gr.Button("입력")
        with gr.Column(scale=1):
            output = gr.Textbox(label="결과", interactive=False)

    submit_btn.click(
        fn=show_character_info,
        inputs=character_input,
        outputs=output,
    )

# 코드 수정 후 다시 실행 할 때는 close() 실행
demo.launch()

In [ ]:
demo.close()

### 여러 이벤트 동시에 처리하기

```bash

gr.on(
    triggers=[submit_btn.click, character_input.submit],
    fn=show_character_info,
    ...
)
```

### 이벤트 핸들러 함수의 입력과 출력 확장
```bash

def show_character_info(character: str) -> tuple[str, str]:
    """입력된 캐릭터 이름을 받아 안내 문구를 반환한다."""

    message = f"{character}에 대해 알아 봅시다"
    return message, ''


gr.on(
    triggers=[submit_btn.click, character_input.submit],
    fn=show_character_info,
    inputs=character_input,
    outputs=[output, character_input],
)
```

### 대화형 Chatbot 화면

챗봇 대화형 (role) - AIMessage => "assistant", HumanMessage => "user" 

```bash

chatbot = gr.Chatbot(label="대화창", height=420)
```

In [ ]:
def show_chat_dialog(user_input: str, chat_history: list[dict[str, str]]) -> tuple[list[dict[str, str]], str]:

    if user_input and user_input.strip():
        chat_history += [{"role": "user", "content": user_input}]
        reply = f"{user_input}에 대해 알아 봅시다"
        chat_history += [{"role": "assistant", "content": reply}]
    else:
        chat_history += [{"role": "assistant", "content": "캐릭터 이름을 입력해 주세요."}]

    return chat_history, ""

with gr.Blocks() as chatbot_demo:

    gr.Markdown("""
        ## 🕵️‍♂️ 명탐정 코난 세계관에 오신 것을 환영합니다.
        ### 명탐정 코난의 등장 인물 중 좋아하는 캐릭터를 입력하세요
    """)

    chatbot = gr.Chatbot(
        label="코난 캐릭터 챗봇",
        height=200
    )
    character_input = gr.Textbox(label="캐릭터 이름 입력")

    # 버튼 이벤트 연결
    character_input.submit(
        fn=show_chat_dialog,
        inputs=[character_input, chatbot],
        outputs=[chatbot, character_input],
    )

chatbot_demo.launch()


In [ ]:
chatbot_demo.close()